## tl;dr
确定性构建结论：`review_required`。本 Notebook 只复核 150 行候选合同与审核门，不运行模型，也不导入基线。


## Context & Methods
输入为冻结的 v1 manifests、绑定 evidence、源 PDF 和 source/evaluator QA。所有题使用同一规则；case_id、难度与生产成败不进入语义审核 payload。

### Key Assumptions
字符串分组、分支切分和页面可提取文本核对只能生成候选，任何审核标记都不能被视为已确认金标。


In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd() if (Path.cwd() / 'qa' / 'summary.json').exists() else Path.cwd() / 'docs' / 'evaluation-baselines' / '2026-08-10-three-difficulty' / 'remediation-2026-08-10' / 'suite-v2-concept-contract'
summary = json.loads((ROOT / 'qa' / 'summary.json').read_text(encoding='utf-8'))
cases = json.loads((ROOT / 'qa' / 'cases.json').read_text(encoding='utf-8'))
len(cases), len({row['parent_case_id'] for row in cases})


## Data
150 行 QA 的 grain 是 parent_case_id；以下检查覆盖、唯一性和脱敏审阅包。


In [ ]:
assert len(cases) == 150
assert len({row['case_id'] for row in cases}) == 150
assert len({row['parent_case_id'] for row in cases}) == 150
assert sum(summary['difficulty'][d]['case_count'] for d in ('easy','medium','hard')) == 150
summary['counts']


## Results
按难度与审核状态检查候选覆盖；精确计数来自已保存 summary，不依赖隐藏状态。


In [ ]:
{difficulty: summary['difficulty'][difficulty] for difficulty in ('easy','medium','hard')}


In [ ]:
requests = [json.loads(line) for line in (ROOT / 'review' / 'requests.jsonl').read_text(encoding='utf-8').splitlines() if line]
assert len(requests) == 150
assert len({row['prompt_sha256'] for row in requests}) == 1
assert all('case_id' not in row['request']['messages'][1]['content'] for row in requests)
len(requests), requests[0]['prompt_sha256']


## Takeaways
- v2 候选与旧 manifests 并存，且 suite 明确禁止导入/登记。
- 自动提取能覆盖全部 150 题，但审核标记代表真实语义不确定性。
- 仅在校准、费用门、resume 和全量同 prompt 审阅完成后，才可生成 reviewed 版本；本产物未执行外呼。
